# News Classification Using BERT 


# Load model and tokenizer

In [ ]:
from transformers import BertTokenizer, AutoModelForSequenceClassification

In [33]:
model_path = "best-model-v1"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
# model.to(device)

In [ ]:
# You can download the best trained model from huggingface 
# model_path = "clhuang/albert-news-classification"
# model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [5]:
# tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
#tokenizer = BertTokenizer.from_pretrained(model_path)

# Define category index

In [26]:
# Category index
news_categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']
idx2cate = { i : cate for i, cate in enumerate(news_categories)}



In [27]:
idx2cate

{0: '政治',
 1: '科技',
 2: '運動',
 3: '證卷',
 4: '產經',
 5: '娛樂',
 6: '生活',
 7: '國際',
 8: '社會',
 9: '文化',
 10: '兩岸'}

# Define prediction function 

In [28]:
import numpy as np

In [35]:
# get category probability
def get_category_proba( text ):
    max_length = 512
    # prepare our text into tokenized sequence
    inputs = tokenizer(text, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    # perform inference to our model
    outputs = model(**inputs)
    # get output probabilities by doing softmax
    probs = outputs[0].softmax(1)

    # executing argmax function to get the candidate label
    # probs.argmax()
    label_index = probs.argmax(dim=1)[0].tolist() # convert tensor to int
    # label_index = np.argmax(probs.detach(), axis=1)
    
    label = idx2cate[ label_index ]

    # Note that result is numpy format and it should be convert to float
    proba = round(float(probs.tolist()[0][label_index]),2)

    response = {'label': label, 'proba': proba}

    return response


# Have a try

In [36]:
text = '俄羅斯2月24日入侵烏克蘭至今不到3個月，芬蘭已準備好扭轉奉行了75年的軍事不結盟政策，申請加入北約。芬蘭總理馬林昨天表示，「希望我們下星期能與瑞典一起提出申請」。'
get_category_proba(text)

{'label': '國際', 'proba': 0.99}

In [40]:
text = "兒童疫苗開始施打"
get_category_proba(text)

{'label': '生活', 'proba': 0.94}

# Step by step demonstration

In [37]:
text = "兒童疫苗開始施打"

# prepare our text into tokenized sequence
max_length = 250
inputs = tokenizer([text], padding=True, truncation=True, max_length=max_length, return_tensors="pt")


In [38]:
inputs

{'input_ids': tensor([[ 101, 1051, 4997, 4554, 5728, 7274, 1993, 3177, 2802,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [39]:
# perform inference to our model
outputs = model(**inputs)


In [14]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-1.0633, -1.5437, -1.3545, -1.6684, -0.5649, -1.2882, 11.4257, -0.9519,
         -0.9584, -0.9701, -1.6157]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [15]:
# get output probabilities by doing softmax
probs = outputs[0].softmax(1)

In [16]:
probs

tensor([[3.7676e-06, 2.3304e-06, 2.8157e-06, 2.0571e-06, 6.2016e-06, 3.0088e-06,
         9.9997e-01, 4.2117e-06, 4.1844e-06, 4.1354e-06, 2.1684e-06]],
       grad_fn=<SoftmaxBackward0>)

In [17]:
probs.tolist()

[[3.7675640669476707e-06,
  2.3303625766857294e-06,
  2.8156625830888515e-06,
  2.057126266663545e-06,
  6.201594715093961e-06,
  3.0087899176578503e-06,
  0.9999651908874512,
  4.211732630210463e-06,
  4.184404588158941e-06,
  4.135417839279398e-06,
  2.168425680793007e-06]]

In [18]:
np.argmax(probs.tolist())

6

In [19]:
probs.argmax(dim=1)

tensor([6])

In [20]:
probs.argmax(dim=1)[0].tolist()

6

In [21]:
# executing argmax function to get the candidate label
# probs.argmax(dim=1)
label_index = probs.argmax(dim=1)[0].tolist()

In [22]:
label_index

6

In [23]:
label = idx2cate[ label_index ]

In [24]:

# Note that result is numpy format and it should be convert to float
# proba = round(float(max(result[0])),2)
proba = round(float(probs.tolist()[0][label_index]),2)

response = {'label': label, 'proba': proba}
response

{'label': '生活', 'proba': 1.0}

# Put them all together for Django

In [ ]:
from transformers import BertTokenizer, AlbertForSequenceClassification
model_path = "clhuang/albert-news-classification"
model = AlbertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

# get category probability
def get_category_proba( text ):
    max_length = 512
    # prepare token sequence
    inputs = tokenizer([text], padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    # perform inference
    outputs = model(**inputs)
    # get output probabilities by doing softmax
    probs = outputs[0].softmax(1)

    # executing argmax function to get the candidate label index
    label_index = probs.argmax(dim=1)[0].tolist() # convert tensor to int
    # get the label name        
    label = idx2cate[ label_index ]

    # get the label probability
    proba = round(float(probs.tolist()[0][label_index]),2)

    response = {'label': label, 'proba': proba}

    return response
 